In [ ]:
!gdown 1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I

Downloading...
From: https://drive.google.com/uc?id=1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I
To: /content/df_train.csv
100% 286k/286k [00:00<00:00, 5.55MB/s]


In [ ]:
!gdown 1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG

Downloading...
From: https://drive.google.com/uc?id=1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG
To: /content/df_test.csv
100% 71.9k/71.9k [00:00<00:00, 4.69MB/s]


In [ ]:
!nvidia-smi

Fri Jun 12 08:49:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd

df = pd.read_csv("df_train.csv")
df

,dwt_mean,dwt_max,dwt_min,dwt_median,dwt_q1,dwt_q3,dwt_iqr,mfcc_mean,mfcc_std,mfcc_max,...,rms_min,rms_median,rms_var,rms_skew,rms_kurtosis,rms_q1,rms_q3,rms_range,shannon_entropy,audio_type
0,1.206750e-08,0.013878,-0.014668,-1.500644e-11,-0.000028,0.000026,0.000055,-81.473040,295.34805,152.37076,...,9.717603e-08,0.000048,1.730892e-07,1.283449,0.308629,1.093666e-05,0.000524,0.001483,8.866710,myocardial
1,5.968118e-07,0.039521,-0.040875,1.670743e-11,-0.000016,0.000017,0.000033,-82.751785,298.57016,158.75189,...,1.647872e-07,0.000034,7.909553e-08,4.731361,22.492230,9.232333e-06,0.000076,0.001755,5.972518,myocardial
2,3.870791e-07,0.032473,-0.023627,7.594341e-12,-0.000017,0.000016,0.000033,-81.980400,297.24650,171.03497,...,1.124997e-07,0.000082,3.880997e-08,3.890662,20.913025,3.342536e-06,0.000215,0.001343,7.658798,myocardial
3,-4.941957e-07,0.131609,-0.118817,-2.006760e-11,-0.000014,0.000014,0.000028,-77.834270,280.35214,168.92433,...,9.554750e-08,0.000026,5.670202e-06,4.185934,17.443203,5.695775e-06,0.000075,0.015121,7.720282,myocardial
4,4.204849e-07,0.002665,-0.002124,-7.600626e-11,-0.000031,0.000029,0.000060,-82.015420,298.52830,97.61485,...,8.227275e-07,0.000036,3.165749e-09,0.953618,-0.323343,1.287166e-05,0.000091,0.000202,9.517833,myocardial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,-1.585129e-08,0.011102,-0.009699,-2.031497e-09,-0.000019,0.000020,0.000039,-82.583570,297.38315,139.96802,...,1.878590e-06,0.000031,8.288504e-09,3.186870,11.402454,1.837468e-05,0.000060,0.000525,7.771392,normal
444,5.693161e-07,0.022825,-0.028649,-4.278836e-11,-0.000013,0.000013,0.000027,-83.153275,299.87430,200.29291,...,1.362070e-07,0.000027,1.346874e-07,3.889586,15.481110,1.185905e-05,0.000067,0.002202,5.310026,normal
445,4.123294e-09,0.060122,-0.060773,-4.331842e-10,-0.000004,0.000004,0.000008,-84.163050,296.94974,180.51411,...,7.595175e-07,0.000011,9.946041e-07,2.224082,3.260276,5.449809e-06,0.000059,0.003769,5.532661,normal
446,-3.339084e-08,0.089756,-0.102157,1.016094e-11,-0.000002,0.000002,0.000004,-82.164860,292.40936,203.16748,...,1.403314e-07,0.000011,1.463136e-06,5.434349,30.395962,8.566282e-07,0.000142,0.008902,6.255238,normal


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, MaxPooling1D, Dense, Dropout, Flatten, LeakyReLU, Input

LEAKY_ALPHA = 0.01

model = Sequential()

input_shape = (51, 1)
num_classes = 2

model.add(Input(shape=input_shape))

model.add(Conv1D(filters=32, kernel_size=1, padding='same'))
model.add(BatchNormalization()) # [cite: 520]
model.add(LeakyReLU(alpha=LEAKY_ALPHA)) # [cite: 533]

model.add(MaxPooling1D(pool_size=2, strides=2))

model.add(Conv1D(filters=32, kernel_size=3, padding='same', groups=32))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))

model.add(Conv1D(filters=64, kernel_size=1, padding='same'))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))

model.add(Flatten())

model.add(Dense(64))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))
model.add(Dropout(0.5))

model.add(Dense(num_classes, activation='softmax'))

opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 51, 32)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 51, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 51, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 25, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 25, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 25, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 25, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 25, 64)         │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 25, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,666 (412.76 KB)

 Trainable params: 105,282 (411.26 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
X = df.drop('audio_type', axis=1).values
y = df['audio_type'].values

# Encode categorical labels to numerical
from sklearn.preprocessing import LabelEncoder, StandardScaler
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y)

# One-hot encode numerical labels
y_one_hot = tf.keras.utils.to_categorical(y_numerical, num_classes=2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for NaN values in features
if np.isnan(X_scaled).any():
    print("Warning: NaN values found in scaled features (X_scaled). This can cause 'nan' loss during training.")
else:
    print("No NaN values found in scaled features (X_scaled).")

No NaN values found in scaled features (X_scaled).


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np

X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fold_no = 1
fold_metrics = {
    'accuracy': [],
    'sensitivity': [],
    'specificity': [],
    'auroc': []
}

print("Starting 10-Fold Cross-Validation on Training Set...")

for train_index, val_index in kfold.split(X_train_full):
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]

    print(f'------------------------------------------------------------------------')
    print(f'Training for Fold {fold_no} ...')

    model.fit(
        X_fold_train,
        y_fold_train,
        batch_size=32,
        epochs=30,
        verbose=0,
        validation_data=(X_fold_val, y_fold_val)
    )

    y_pred_probs = model.predict(X_fold_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_fold_val, axis=1)

    acc = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    auroc = roc_auc_score(y_fold_val[:, 1], y_pred_probs[:, 1])

    fold_metrics['accuracy'].append(acc)
    fold_metrics['sensitivity'].append(sens)
    fold_metrics['specificity'].append(spec)
    fold_metrics['auroc'].append(auroc)

    print(f'Fold {fold_no} Results: Acc={acc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}, AUROC={auroc:.4f}')
    fold_no += 1

# Summary
print(f'\n------------------------------------------------------------------------')
print(f'Summary Results (Mean ± STD) over 10 Folds:')
for metric, values in fold_metrics.items():
    print(f'{metric.capitalize()}: {np.mean(values)*100:.2f}% ± {np.std(values)*100:.2f}%')

Starting 10-Fold Cross-Validation on Training Set...
------------------------------------------------------------------------
Training for Fold 1 ...
Fold 1 Results: Acc=0.7561, Sens=0.6667, Spec=0.8261, AUROC=0.7826
------------------------------------------------------------------------
Training for Fold 2 ...
Fold 2 Results: Acc=0.9268, Sens=1.0000, Spec=0.8800, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 3 ...
Fold 3 Results: Acc=0.9756, Sens=1.0000, Spec=0.9524, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 4 ...
Fold 4 Results: Acc=0.9750, Sens=1.0000, Spec=0.9412, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 5 ...
Fold 5 Results: Acc=0.9750, Sens=1.0000, Spec=0.9474, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 6 ...
Fold 6 Results: Acc=1.0000, Se

In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 77.78%
Sensitivity: 72.73%
Specificity: 82.61%
AUROC: 0.8518


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 77.78%
Sensitivity: 77.27%
Specificity: 78.26%
AUROC: 0.8735


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 84.44%
Sensitivity: 86.36%
Specificity: 82.61%
AUROC: 0.9061


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 75.56%
Sensitivity: 72.73%
Specificity: 78.26%
AUROC: 0.8607


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 82.22%
Sensitivity: 81.82%
Specificity: 82.61%
AUROC: 0.8735


In [ ]:
df_test = pd.read_csv('df_test.csv')

In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0
specificity_test = tn_test / (tn_test + fp_test) if (tn_test + fp_test) > 0 else 0

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

--- Evaluation on df_test --- 
Test Accuracy: 76.79%
Test Sensitivity: 80.36%
Test Specificity: 73.21%
Test AUROC: 0.8630
Confusion Matrix:
[[41 15]
 [11 45]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0
specificity_test = tn_test / (tn_test + fp_test) if (tn_test + fp_test) > 0 else 0

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

--- Evaluation on df_test --- 
Test Accuracy: 75.89%
Test Sensitivity: 75.00%
Test Specificity: 76.79%
Test AUROC: 0.8385
Confusion Matrix:
[[43 13]
 [14 42]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0
specificity_test = tn_test / (tn_test + fp_test) if (tn_test + fp_test) > 0 else 0

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step

--- Evaluation on df_test --- 
Test Accuracy: 75.89%
Test Sensitivity: 75.00%
Test Specificity: 76.79%
Test AUROC: 0.8385
Confusion Matrix:
[[43 13]
 [14 42]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0
specificity_test = tn_test / (tn_test + fp_test) if (tn_test + fp_test) > 0 else 0

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step

--- Evaluation on df_test --- 
Test Accuracy: 77.68%
Test Sensitivity: 80.36%
Test Specificity: 75.00%
Test AUROC: 0.8557
Confusion Matrix:
[[42 14]
 [11 45]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0
specificity_test = tn_test / (tn_test + fp_test) if (tn_test + fp_test) > 0 else 0

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

--- Evaluation on df_test --- 
Test Accuracy: 78.57%
Test Sensitivity: 78.57%
Test Specificity: 78.57%
Test AUROC: 0.8619
Confusion Matrix:
[[44 12]
 [12 44]]
